In [ ]:
# Model
import os
from typing import Any

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from langchain_ollama import ChatOllama


class TokenCountsCallbackHandler(BaseCallbackHandler):
    def on_llm_end(self, response: LLMResult, **_: Any) -> None:
        for generations in response.generations:
            for generation in generations:
                message = getattr(generation, "message", None)
                if message is not None:
                    meta = getattr(message, "usage_metadata", None)
                    if meta:
                        print(
                            f"🪙 Tokens input: {meta.get('input_tokens', 'N/A')}, "
                            f"Tokens output: {meta.get('output_tokens', 'N/A')}, "
                            f"Tokens total: {meta.get('total_tokens', 'N/A')}"
                        )


token_counts_callback_handler = TokenCountsCallbackHandler()


def make_llm(temperature: float) -> ChatOllama:
    return ChatOllama(
        base_url=os.environ.get("OLLAMA_HOST", "http://localhost:11434"),
        keep_alive="12m",
        model="gemma4:e2b",
        temperature=temperature,
        reasoning=False,
        callbacks=[token_counts_callback_handler],
    )


llm_validator = make_llm(temperature=0.06)
llm_analytical = make_llm(temperature=0.12)
llm_generative = make_llm(temperature=0.24)
llm_creative = make_llm(temperature=0.48)

hello_world = llm_creative.invoke("Hello, world!")
hello_world.pretty_print()

In [ ]:
# Search Sources
from typing import NotRequired, TypedDict

import trafilatura
from langchain_community.utilities import SearxSearchWrapper
from langgraph.graph import END, START, StateGraph

from src.prompts import search_sources_prompt_template
from src.subjects import Subject

SEARXNG_HOST = os.environ.get("SEARXNG_HOST", "http://localhost:8889")

# gemma4:e2b has a 128k token context window (~4 chars/token).
# 12 sources × 24 000 chars = 288 000 chars ≈ 72 000 tokens, leaving
# a comfortable margin for prompt overhead and output.
MAX_CHARS_PER_SOURCE = 32_000


class SubjectState(TypedDict):
    subject: Subject
    query: NotRequired[str]
    sources: NotRequired[str]


def search_sources(state: SubjectState) -> SubjectState:
    subject = state.get("subject")
    print(f"🔍 Searching sources for subject: {subject.name}...")
    query = search_sources_prompt_template.format(
        subject_name=subject.name,
        category_name=subject.category.name,
    )

    searxng_wrapper = SearxSearchWrapper(searx_host=SEARXNG_HOST)
    searxng_results = searxng_wrapper.results(query, num_results=12)

    trafilatura_results = []
    for i, searxng_result in enumerate(searxng_results, start=1):
        url = searxng_result.get("link", "")
        title = searxng_result.get("title", "").strip()
        text = trafilatura.extract(
            trafilatura.fetch_url(url),
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )
        if not text:
            print(f"⚠️ Warning: No text extracted from URL: {url}")
            continue
        if len(text) > MAX_CHARS_PER_SOURCE:
            text = text[:MAX_CHARS_PER_SOURCE]
            print(f'✂️ Truncated source {i} "{title}" to {MAX_CHARS_PER_SOURCE} chars: {url}')
        heading = title if title else url
        trafilatura_results.append(
            f"## Search Result {i}: {heading}\n\n**Source:** [{url}]({url})\n\n{text}"
        )

    if not trafilatura_results:
        raise ValueError(f"No content could be extracted for query: {query}")

    sources = "\n\n---\n\n".join(trafilatura_results)

    return {
        "subject": subject,
        "query": query,
        "sources": sources,
    }


graph_builder = StateGraph(SubjectState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", END)
graph = graph_builder.compile()
# display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# print(state)

In [ ]:
# Generate Document from Sources
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser

from src.output import write_affirmations_document_markdown
from src.prompts import affirmations_generate_document_from_sources_prompt_template

inflect_engine = inflect.engine()

str_output_parser = StrOutputParser()


class GenerateDocumentFromSourcesState(SubjectState):
    document: NotRequired[str]


def generate_document(
    state: GenerateDocumentFromSourcesState,
) -> dict[str, str]:
    print(f"📄 Generating document from sources for subject: {state.get('subject').name}...")
    subject = state.get("subject")
    sources = state.get("sources")

    messages = affirmations_generate_document_from_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        sources=sources,
    )

    document = str_output_parser.invoke(llm_generative.invoke(messages))
    write_affirmations_document_markdown(subject, document)

    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromSourcesState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "generate_document")
graph_builder.add_edge("generate_document", END)
graph = graph_builder.compile()
# display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# print(state)

In [ ]:
# Generate Document from Source Analysis
from typing import NotRequired

import inflect
from langchain_core.output_parsers import StrOutputParser
from tenacity import RetryCallState, Retrying, stop_after_attempt, wait_exponential

from src.prompts import (
    affirmations_analyze_sources_prompt_template,
    affirmations_generate_document_prompt_template,
)

inflect_engine = inflect.engine()

str_output_parser = StrOutputParser()


def before_sleep(retry_call_state: RetryCallState) -> None:
    exception = retry_call_state.outcome.exception() if retry_call_state.outcome else None
    delay = retry_call_state.next_action.sleep if retry_call_state.next_action else 0
    print(
        f"⚠️ Attempt {retry_call_state.attempt_number} failed: {exception}. Retrying in {delay:.1f}s..."
    )


def attempts() -> Retrying:
    return Retrying(
        stop=stop_after_attempt(6),
        wait=wait_exponential(multiplier=1, min=1, max=10),
        before_sleep=before_sleep,
        reraise=True,
    )


class GenerateDocumentFromAnalysisState(GenerateDocumentFromSourcesState):
    source_analysis: NotRequired[str]


def analyze_sources(state: GenerateDocumentFromAnalysisState) -> dict[str, str]:
    subject = state.get("subject")
    print(f"💭 Analyzing sources for subject: {subject.name}")

    messages = affirmations_analyze_sources_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        sources=state.get("sources"),
    )

    source_analysis: str | None = None
    for attempt in attempts():
        with attempt:
            source_analysis = str_output_parser.invoke(llm_analytical.invoke(messages))
    assert source_analysis is not None, "Failed to analyze sources after multiple attempts."

    return {"source_analysis": source_analysis}


def generate_document(
    state: GenerateDocumentFromAnalysisState,
) -> dict[str, str]:
    subject = state.get("subject")
    print(f"📄 Generating document from source analysis of subject: {subject.name}...")

    messages = affirmations_generate_document_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        category_name_plural=inflect_engine.plural(subject.category.name),  # type: ignore[arg-type]
        source_analysis=state.get("source_analysis", ""),
    )

    document = str_output_parser.invoke(llm_generative.invoke(messages))
    write_affirmations_document_markdown(subject, document)
    return {"document": document}


graph_builder = StateGraph(GenerateDocumentFromAnalysisState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("analyze_sources", analyze_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "analyze_sources")
graph_builder.add_edge("analyze_sources", "generate_document")
graph_builder.add_edge("generate_document", END)
graph = graph_builder.compile()
# display(Image(graph.get_graph().draw_mermaid_png()))

# state = graph.invoke({"subject": TAROT_CARDS[0]})
# pprint(state)

In [ ]:
# Generate Affirmations
import operator
import re
from typing import Annotated, NotRequired, cast

import inflect
import json5 as _json5
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from pydantic import BaseModel as PydanticBaseModel

from src.grammars import GRAMMARS, Grammar
from src.models import (
    Affirmation,
    GeneratedAffirmations,
    GrammarAffirmations,
    SubjectAffirmations,
    ValidationResult,
)
from src.output import write_affirmations_json, write_affirmations_markdown
from src.prompts import (
    affirmations_analyze_document_prompt_template,
    affirmations_generate_affirmations_prompt_template,
    affirmations_validate_affirmation_prompt_template,
    affirmations_validate_affirmation_subject_prompt_template,
)
from src.subjects import (
    SUBJECTS,
)

inflect_engine = inflect.engine()
str_output_parser = StrOutputParser()


class Json5PydanticOutputParser(BaseOutputParser[PydanticBaseModel]):
    """Output parser using json5 to handle lenient JSON (e.g. trailing commas)."""

    pydantic_object: type[PydanticBaseModel]

    def parse(self, text: str) -> PydanticBaseModel:
        try:
            # Strip markdown code fences (e.g. ```json\n...\n```)
            stripped = re.sub(
                r"^```(?:\w+)?\s*\n?(.*?)\n?```\s*$", r"\1", text.strip(), flags=re.DOTALL
            )
            data = _json5.loads(stripped)
            return self.pydantic_object.model_validate(data)
        except Exception as e:
            raise OutputParserException(f"Invalid json output: {text}") from e

    @property
    def _type(self) -> str:
        return "json5_pydantic"


class GenerateAffirmationsState(GenerateDocumentFromAnalysisState):
    document_analysis: NotRequired[str]
    grammars: list[Grammar]
    grammar_affirmations: NotRequired[list[GrammarAffirmations]]
    validation_failures: NotRequired[Annotated[list[str], operator.add]]


def analyze_document(state: GenerateAffirmationsState) -> dict[str, str]:
    subject = state.get("subject")
    print(f"💭 Analyzing document for subject: {subject.name}...")

    messages = affirmations_analyze_document_prompt_template.format_messages(
        subject_name=subject.name,
        category_name=subject.category.name,
        document=state.get("document", ""),
    )

    document_analysis = str_output_parser.invoke(llm_analytical.invoke(messages))
    return {"document_analysis": document_analysis}


def generate_affirmations(
    state: GenerateAffirmationsState,
) -> dict[str, list[GrammarAffirmations]]:
    subject = state.get("subject")
    print(f"🔮 Generating affirmations for subject: {subject.name}...")

    chain = llm_creative.bind(
        format=GeneratedAffirmations.model_json_schema()
    ) | Json5PydanticOutputParser(pydantic_object=GeneratedAffirmations)
    grammar_affirmations: list[GrammarAffirmations] = []

    for grammar in state.get("grammars", []):
        print(f"🔮 Generating affirmations for grammar: {grammar.name}...")

        messages = affirmations_generate_affirmations_prompt_template.format_messages(
            subject_name=subject.name,
            document_analysis=state.get("document_analysis", ""),
            grammar_name=grammar.name,
            grammar_description=grammar.description,
            grammar_specifier_descriptions=grammar.specifier_descriptions,
            grammar_specifiers=", ".join(grammar.specifiers),
            grammar_examples="; ".join(grammar.examples),
            grammar_emoji=grammar.emoji,
        )

        generated: GeneratedAffirmations | None = None
        for attempt in attempts():
            with attempt:
                generated = cast("GeneratedAffirmations", chain.invoke(messages))
        assert generated is not None, "Failed to generate affirmations after multiple attempts."

        affirmations = [
            Affirmation(text=re.sub(r"^[^\w\s]+|[^\w\s!?]+$", "", text).strip())
            for text in generated.affirmations
        ]
        grammar_affirmations.append(GrammarAffirmations(grammar=grammar, affirmations=affirmations))

        affirmations_text = "\n".join(f"- {affirmation.text}" for affirmation in affirmations)
        print(
            f"🔮 Generated {len(affirmations)} affirmations for grammar: {grammar.name}\n{affirmations_text}"
        )

    return {"grammar_affirmations": grammar_affirmations}


def validate_affirmations_grammars(
    state: GenerateAffirmationsState,
) -> dict[str, list[str]]:
    subject = state.get("subject")
    print(f"☑️ Validating affirmations grammar for subject: {subject.name}...")

    chain = llm_validator.bind(
        format=ValidationResult.model_json_schema()
    ) | Json5PydanticOutputParser(pydantic_object=ValidationResult)
    failures: list[str] = []

    for grammar_affirmations in state.get("grammar_affirmations", []):
        grammar = grammar_affirmations.grammar
        print(f"☑️ Validating affirmations grammar for grammar: {grammar.name}...")

        for affirmation in grammar_affirmations.affirmations:
            print(f"☑️ Validating affirmation grammar: '{affirmation.text}'...")

            messages = affirmations_validate_affirmation_prompt_template.format_messages(
                affirmation_text=affirmation.text,
                grammar_name=grammar.name,
                grammar_description=grammar.description,
                grammar_specifiers=", ".join(grammar.specifiers),
                grammar_specifier_descriptions=grammar.specifier_descriptions,
                grammar_examples="; ".join(grammar.examples),
            )

            result: ValidationResult | None = None
            for attempt in attempts():
                with attempt:
                    result = cast("ValidationResult", chain.invoke(messages))
            assert result is not None, "Failed to validate affirmation after multiple attempts."

            if not result.valid:
                failure = f"[{grammar.name}] '{affirmation.text}': {result.reason}"
                failures.append(failure)
                print(f"❌ Invalid affirmation grammar: {failure}")
            else:
                print(f"✅ Validated affirmation grammar: '{affirmation.text}'")

    if not failures:
        print("🎉 Validated affirmations grammar")
    else:
        print(f"⛔ Invalid affirmations grammar: {len(failures)}")

    return {"validation_failures": failures}


def validate_affirmations_subject(
    state: GenerateAffirmationsState,
) -> dict[str, list[str]]:
    subject = state.get("subject")
    print(f"☑️ Validating affirmations subject for subject: {subject.name}...")

    chain = llm_validator.bind(
        format=ValidationResult.model_json_schema()
    ) | Json5PydanticOutputParser(pydantic_object=ValidationResult)
    failures: list[str] = []

    for grammar_affirmations in state.get("grammar_affirmations", []):
        grammar = grammar_affirmations.grammar
        print(f"☑️ Validating affirmations subject for grammar: {grammar.name}...")

        for affirmation in grammar_affirmations.affirmations:
            print(f"☑️ Validating affirmation subject: '{affirmation.text}'...")
            messages = affirmations_validate_affirmation_subject_prompt_template.format_messages(
                affirmation_text=affirmation.text,
                subject_name=subject.name,
                category_name=subject.category.name,
                document=state.get("document", ""),
            )

            result: ValidationResult | None = None
            for attempt in attempts():
                with attempt:
                    result = cast("ValidationResult", chain.invoke(messages))
            assert result is not None, "Failed to validate affirmation after multiple attempts."

            if not result.valid:
                failure = f"[{grammar.name}] '{affirmation.text}': {result.reason}"
                failures.append(failure)
                print(f"❌ Invalid affirmation subject: {failure}")
            else:
                print(f"✅ Validated affirmation subject: '{affirmation.text}'")

    if not failures:
        print("🎉 Validated affirmations subjects")
    else:
        print(f"⛔ Invalid affirmations subjects: {len(failures)}.")

    return {"validation_failures": failures}


def write_affirmations(state: GenerateAffirmationsState) -> GenerateAffirmationsState:
    subject_affirmations = SubjectAffirmations(
        subject=state.get("subject"),
        grammars=state.get("grammar_affirmations", []),
    )
    write_affirmations_json(subject_affirmations)
    write_affirmations_markdown(subject_affirmations)
    return state


graph_builder = StateGraph(GenerateAffirmationsState)
graph_builder.add_node("search_sources", search_sources)
graph_builder.add_node("analyze_sources", analyze_sources)
graph_builder.add_node("generate_document", generate_document)
graph_builder.add_node("analyze_document", analyze_document)
graph_builder.add_node("generate_affirmations", generate_affirmations)
graph_builder.add_node("validate_affirmations_grammars", validate_affirmations_grammars)
graph_builder.add_node("validate_affirmations_subject", validate_affirmations_subject)
graph_builder.add_node("write_affirmations", write_affirmations)
graph_builder.add_edge(START, "search_sources")
graph_builder.add_edge("search_sources", "analyze_sources")
graph_builder.add_edge("analyze_sources", "generate_document")
graph_builder.add_edge("generate_document", "analyze_document")
graph_builder.add_edge("analyze_document", "generate_affirmations")
graph_builder.add_edge("generate_affirmations", "validate_affirmations_grammars")
graph_builder.add_edge("validate_affirmations_grammars", "validate_affirmations_subject")
graph_builder.add_edge("validate_affirmations_subject", "write_affirmations")
graph_builder.add_edge("write_affirmations", END)
graph = graph_builder.compile()
# display(Image(graph.get_graph().draw_mermaid_png()))


for subject in SUBJECTS:
    state = graph.invoke({"subject": subject, "grammars": GRAMMARS})

    print("\n=== SOURCE ANALYSIS ===")
    print(state.get("source_analysis"))

    print("\n=== DOCUMENT ===")
    print(state.get("document", ""))

    print("\n=== DOCUMENT ANALYSIS ===")
    print(state.get("document_analysis"))

    print("\n=== GENERATED AFFIRMATIONS ===")
    for grammar_affirmation in state.get("grammar_affirmations", []):
        print(f"\n{grammar_affirmation.grammar.name}:")
        for affirmation in grammar_affirmation.affirmations:
            print(f"  - {affirmation.text}")

    print("\n=== INVALID AFFIRMATIONS ===")
    for failure in state.get("validation_failures", []):
        print(f"  {failure}")
    if not state.get("validation_failures"):
        print("  None")